PIFOCast - Dataset generation
==============================
Download ERA5 data, prepare grid definition, and pre-compute all features as .npy cache.

In [1]:
import os
import math
import numpy as np

import xarray as xr
import matplotlib.pyplot as plt

import importlib

import pifocast
importlib.reload(pifocast)

from pifocast import LatLonGrid, create_edge_features, create_position_features

### Data retrieval from ERA5
Requires `~/.cdsapirc` with a valid CDS API key.
Uncomment to download.

In [2]:
# import cdsapi
#
# years = ["2024"]
# monthes = ["01", "02", "03", "04", "05", "06",
#            "07", "08", "09", "10", "11", "12"]
# days = [str(d).zfill(2) for d in range(1, 32)]
# times = ["00:00", "06:00"]
#
# for year in years:
#     for month in monthes:
#         request = {
#             "product_type": ["reanalysis"],
#             "variable": ["geopotential",
#                         "u_component_of_wind",
#                         "v_component_of_wind"],
#             "year": [year],
#             "month": [month],
#             "day": days,
#             "time": times,
#             "pressure_level": ["500"],
#             "data_format": "grib",
#             "download_format": "unarchived"
#         }
#         target = f'dataset/{year}{month}.grib'
#         client = cdsapi.Client()
#         client.retrieve("reanalysis-era5-pressure-levels", request, target)

### Grid initialization
Load a sample GRIB file to define the lat/lon grid geometry.

In [3]:
grid = LatLonGrid("dataset/202504.grib", regrid_factor=4)
print(f"NLat={grid.NLat}, NLon={grid.NLon}, NNodes={grid.NNodes}, NEdges={grid.NEdges}")

NLat=181, NLon=360, NNodes=65160, NEdges=257760


### Save pre-computed static features
Edge and position features are static for a given grid.

In [4]:
edge_features = create_edge_features(grid)
pos_features = create_position_features(grid)
sources = np.array(grid.sources, dtype=np.int32)
targets = np.array(grid.targets, dtype=np.int32)

np.save("dataset/edge_features.npy", edge_features)
np.save("dataset/pos_features.npy", pos_features)
np.save("dataset/sources.npy", sources)
np.save("dataset/targets.npy", targets)
print("Saved edge_features, pos_features, sources, targets to dataset/")
print(f"edge_features shape: {edge_features.shape}")
print(f"pos_features shape: {pos_features.shape}")

Saved edge_features, pos_features, sources, targets to dataset/
edge_features shape: (257760, 3)
pos_features shape: (65160, 3)


### Pre-compute all examples as .npy cache
Parse each GRIB file once, collect all (node_features, labels) pairs,
and save them as per-month .npy arrays. This avoids re-parsing GRIB
at every training epoch.

In [5]:
from pifocast import pifoGridGenerator

trainFiles = [
    "202401.grib", "202402.grib", "202403.grib",
    "202404.grib", "202405.grib", "202406.grib",
    "202407.grib", "202408.grib", "202409.grib",
    "202410.grib", "202411.grib", "202412.grib"
]

all_counts = []
for fname in trainFiles:
    path = f"dataset/{fname}"
    month = fname.split(".")[0]
    print(f"Processing {path}...")

    buf_nf, buf_lb = [], []
    for nf, ef, src, tgt, labels in pifoGridGenerator(path, grid, step=2):
        buf_nf.append(nf)
        buf_lb.append(labels)

    all_nf = np.stack(buf_nf, axis=0)
    all_labels = np.stack(buf_lb, axis=0)
    np.save(f"dataset/{month}_nf.npy", all_nf)
    np.save(f"dataset/{month}_labels.npy", all_labels)
    all_counts.append(len(buf_nf))
    print(f"  {month}: {len(buf_nf)} examples saved ({all_nf.shape}, {all_labels.shape})")

print(f"\nTotal: {sum(all_counts)} examples across {len(trainFiles)} files")
print(f"Per-month counts: {all_counts}")

Processing dataset/202401.grib...
    grib index  0
    grib index  2
    grib index  4
    grib index  6
    grib index  8
    grib index  10
    grib index  12
    grib index  14
    grib index  16
    grib index  18
    grib index  20
    grib index  22
    grib index  24
    grib index  26
    grib index  28
    grib index  30
    grib index  32
    grib index  34
    grib index  36
    grib index  38
    grib index  40
    grib index  42
    grib index  44
    grib index  46
    grib index  48
    grib index  50
    grib index  52
    grib index  54
    grib index  56
    grib index  58
    grib index  60
  202401: 31 examples saved ((31, 65160, 6), (31, 65160, 3))
Processing dataset/202402.grib...
    grib index  0
    grib index  2
    grib index  4
    grib index  6
    grib index  8
    grib index  10
    grib index  12
    grib index  14
    grib index  16
    grib index  18
    grib index  20
    grib index  22
    grib index  24
    grib index  26
    grib index  28
    gri

### Verify cached arrays
Quick sanity check that the saved files load correctly.

In [6]:
test_nf = np.load("dataset/202401_nf.npy")
test_lb = np.load("dataset/202401_labels.npy")
print(f"202401_nf: {test_nf.shape}, 202401_labels: {test_lb.shape}")
print(f"Sample node_features[0]: mean={test_nf[0].mean():.4f}, std={test_nf[0].std():.4f}")

202401_nf: (31, 65160, 6), 202401_labels: (31, 65160, 3)
Sample node_features[0]: mean=0.5197, std=0.2836
